# OfficeQA Retrieval Evaluation

Runs semantic top-k retrieval for every question in `data/officeqa/officeqa_pro.csv` using the same `SemTopKOperator` harness pattern as `run_one_retrieval.py`, then reports aggregate precision, recall, and F1.

In [11]:
import csv
import glob
import os

import pandas as pd

REPO_ROOT = os.getcwd()
if not os.path.exists(os.path.join(REPO_ROOT, "pyproject.toml")):
    REPO_ROOT = os.path.abspath(os.path.join(REPO_ROOT, "../.."))

from dotenv import load_dotenv
load_dotenv(os.path.join(REPO_ROOT, ".env"))

from carnot.data.dataset import Dataset
from carnot.operators.sem_topk import SemTopKOperator


QUERIES_CSV = os.path.join(REPO_ROOT, "data/officeqa/officeqa_pro.csv")
DOCS_DIR = os.path.join(REPO_ROOT, "data/officeqa/treasury_bulletins_parsed/transformed")

K = 5
INDEX = "faiss"
EMBEDDING_MODEL = "openai/text-embedding-3-large"
MAX_QUERIES = None
MAX_DOCS = None

In [12]:
with open(QUERIES_CSV, newline="") as queries_file:
    queries = list(csv.DictReader(queries_file))

if MAX_QUERIES is not None:
    queries = queries[:MAX_QUERIES]

document_paths = sorted(glob.glob(os.path.join(DOCS_DIR, "*.txt")))
if MAX_DOCS is not None:
    gold_files = []
    for query in queries:
        gold_files.extend(
            source_file.strip()
            for source_file in query["source_files"].splitlines()
            if source_file.strip()
        )
    gold_paths = [os.path.join(DOCS_DIR, source_file) for source_file in gold_files]
    document_paths = list(dict.fromkeys(gold_paths + document_paths))[:MAX_DOCS]
    document_paths = [path for path in document_paths if os.path.exists(path)]

if not document_paths:
    raise ValueError(f"No .txt documents found in {DOCS_DIR}")

items = []
for document_path in document_paths:
    with open(document_path, errors="replace") as document:
        items.append(
            {
                "uri": document_path,
                "source_file": os.path.basename(document_path),
                "contents": document.read(),
            }
        )

dataset = Dataset(
    name="OfficeQA Documents",
    annotation="Parsed Treasury Bulletin documents for OfficeQA retrieval.",
    items=items,
    dataset_id="officeqa_documents",
)

llm_config = {
    "GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY"),
    "GEMINI_API_KEY": os.getenv("GEMINI_API_KEY"),
}

print(f"Loaded {len(queries)} queries")
print(f"Loaded {len(items)} documents from {DOCS_DIR}")
print(f"Retrieval config: index={INDEX}, k={K}, embedding_model={EMBEDDING_MODEL}")

Loaded 133 queries
Loaded 697 documents from /home/gerardo/carnot/data/officeqa/treasury_bulletins_parsed/transformed
Retrieval config: index=faiss, k=5, embedding_model=openai/text-embedding-3-large


In [16]:
import carnot 
def retrieve_question(question):
    predicted_files = []
    items_out = 0
    error = None
    try:
        # operator = SemTopKOperator(
        #     task=question,
        #     k=K,
        #     dataset_id="RetrievedOfficeQADocuments",
        #     max_workers=1,
        #     index_name=INDEX,
        #     # model_id='gemini/gemini-embedding-2',
        #     model_id=EMBEDDING_MODEL,
        #     llm_config={"GOOGLE_API_KEY": os.getenv("GOOGLE_API_KEY"),
        #                 "GEMINI_API_KEY": os.getenv("GEMINI_API_KEY"),
        #                 "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY")},
        # )
        # output_datasets, stats = operator(dataset.name, {dataset.name: dataset})
        # retrieved = output_datasets["RetrievedOfficeQADocuments"].items
        # predicted_files = [item.get("source_file") for item in retrieved]
        # items_out = stats.items_out
        execution = carnot.Execution(
            query = "Find documents that are necessary to answer the question: " + question,
            datasets=[dataset.name],
            llm_config={
                "model": "gemini/gemini-embedding-2",
                "api_key": os.getenv("GEMINI_API_KEY"),
            }
        )


    except Exception as exc:
        error = repr(exc)
    return predicted_files, items_out, error


In [14]:
# import litellm
# litellm._turn_on_debug()

rows = []
for idx, query in enumerate(queries, start=1):
    predicted_files, items_out, error = retrieve_question(query['question'])

    gold_files = [source_file.strip() for source_file in query["source_files"].splitlines() if source_file.strip()]
    gold = set(gold_files)
    predicted = set(predicted_files).difference({None, ""})

    true_positives = len(gold & predicted)
    precision = true_positives / len(predicted) if predicted else 0.0
    recall = true_positives / len(gold) if gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    rows.append(
        {
            "uid": query["uid"],
            "difficulty": query.get("difficulty"),
            "question": query["question"],
            "answer": query.get("answer"),
            "gold_source_files": gold_files,
            "predicted_source_files": predicted_files,
            "true_positives": true_positives,
            "gold_count": len(gold),
            "predicted_count": len(predicted),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "hit_any": true_positives > 0,
            "items_in": len(items),
            "items_out": items_out,
            "error": error,
        }
    )
    if idx == 1 or idx % 10 == 0 or idx == len(queries):
        print(f"Completed {idx}/{len(queries)} queries")

results_df = pd.DataFrame(rows)
results_df.head()

Completed 1/133 queries
Completed 10/133 queries
Completed 20/133 queries
Completed 30/133 queries
Completed 40/133 queries
Completed 50/133 queries
Completed 60/133 queries
Completed 70/133 queries
Completed 80/133 queries
Completed 90/133 queries
Completed 100/133 queries
Completed 110/133 queries
Completed 120/133 queries
Completed 130/133 queries
Completed 133/133 queries


,uid,difficulty,question,answer,gold_source_files,predicted_source_files,true_positives,gold_count,predicted_count,precision,recall,f1,hit_any,items_in,items_out,error
0,UID0001,hard,What were the total expenditures (in millions ...,"2,602",[treasury_bulletin_1941_01.txt],"[treasury_bulletin_1941_04.txt, treasury_bulle...",0,1,5,0.0,0.0,0.0,False,697,5,None
1,UID0003,hard,Using specifically only the reported values fo...,"44,463",[treasury_bulletin_1954_02.txt],"[treasury_bulletin_1953_07.txt, treasury_bulle...",0,1,5,0.0,0.0,0.0,False,697,5,None
2,UID0004,hard,Using specifically only the reported values fo...,1608.80%,"[treasury_bulletin_1941_01.txt, treasury_bulle...","[treasury_bulletin_1943_01.txt, treasury_bulle...",0,2,5,0.0,0.0,0.0,False,697,5,None
3,UID0005,hard,Using specifically only the reported values fo...,39482.03,"[treasury_bulletin_1941_01.txt, treasury_bulle...","[treasury_bulletin_1953_07.txt, treasury_bulle...",0,2,5,0.0,0.0,0.0,False,697,5,None
4,UID0007,hard,According to the US Treasury's breakdown of bu...,4962.46,[treasury_bulletin_1950_02.txt],"[treasury_bulletin_1945_05.txt, treasury_bulle...",0,1,5,0.0,0.0,0.0,False,697,5,None


In [15]:
total_tp = int(results_df["true_positives"].sum())
total_predicted = int(results_df["predicted_count"].sum())
total_gold = int(results_df["gold_count"].sum())
micro_precision = total_tp / total_predicted if total_predicted else 0.0
micro_recall = total_tp / total_gold if total_gold else 0.0
micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if micro_precision + micro_recall else 0.0

aggregate_metrics = pd.DataFrame(
    [
        {
            "aggregation": "macro_avg_across_queries",
            "queries": len(results_df),
            "precision": results_df["precision"].mean(),
            "recall": results_df["recall"].mean(),
            "f1": results_df["f1"].mean(),
            "hit_rate": results_df["hit_any"].mean(),
            "true_positives": total_tp,
            "gold_files": total_gold,
            "predicted_files": total_predicted,
            "failed_queries": int(results_df["error"].notna().sum()),
            "k": K,
            "index": INDEX,
        },
        {
            "aggregation": "micro_total_across_files",
            "queries": len(results_df),
            "precision": micro_precision,
            "recall": micro_recall,
            "f1": micro_f1,
            "hit_rate": results_df["hit_any"].mean(),
            "true_positives": total_tp,
            "gold_files": total_gold,
            "predicted_files": total_predicted,
            "failed_queries": int(results_df["error"].notna().sum()),
            "k": K,
            "index": INDEX,
        },
    ]
)

aggregate_metrics.style.format(
    {
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "f1": "{:.4f}",
        "hit_rate": "{:.4f}",
    }
)

,aggregation,queries,precision,recall,f1,hit_rate,true_positives,gold_files,predicted_files,failed_queries,k,index
0,macro_avg_across_queries,133,0.0391,0.1115,0.0548,0.1579,26,242,665,0,5,faiss
1,micro_total_across_files,133,0.0391,0.1074,0.0573,0.1579,26,242,665,0,5,faiss


In [ ]:
best_result = results_df.sort_values(
    ["f1", "recall", "precision", "true_positives"],
    ascending=[False, False, False, False],
).head(1)

best_result[
    [
        "uid",
        "difficulty",
        "precision",
        "recall",
        "f1",
        "hit_any",
        "question",
        "gold_source_files",
        "predicted_source_files",
        "answer",
        "error",
    ]
]

,uid,difficulty,precision,recall,f1,hit_any,question,gold_source_files,predicted_source_files,answer,error
0,UID0001,hard,0.0,0.0,0.0,False,What were the total expenditures (in millions ...,[treasury_bulletin_1941_01.txt],[],"2,602",AssertionError()


In [ ]:
worst_result = results_df.sort_values(
    ["f1", "recall", "precision", "true_positives"],
    ascending=[True, True, True, True],
).head(1)

worst_result[
    [
        "uid",
        "difficulty",
        "precision",
        "recall",
        "f1",
        "hit_any",
        "question",
        "gold_source_files",
        "predicted_source_files",
        "answer",
        "error",
    ]
]

,uid,difficulty,precision,recall,f1,hit_any,question,gold_source_files,predicted_source_files,answer,error
0,UID0001,hard,0.0,0.0,0.0,False,What were the total expenditures (in millions ...,[treasury_bulletin_1941_01.txt],[],"2,602",AssertionError()
